# 1. How Whisper hears

Whisper is OpenAI's speech recognition model. Before fine-tuning it, you need to know what goes in, what comes out, and how to measure whether the output is right. In this notebook you will:

1. Look at audio as numbers (a waveform).
2. Turn it into the log-mel spectrogram that Whisper actually sees.
3. Look at the tokens Whisper reads and writes, including its special control tokens.
4. Transcribe audio, first with `generate()` and then with your own decoding loop.
5. Measure accuracy with word error rate (WER) on 73 LibriSpeech clips, and compare two model sizes.

Cells marked **TODO(you)** are yours to write. Most are followed by a check cell that tells you whether your code works.

**Setup:** select the Python kernel from `whisper-finetune/.venv` (top right of the notebook). The first run downloads the model (about 150 MB) and the audio clips.

In [ ]:
import io
import time

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import transformers
from datasets import Audio, load_dataset
from IPython.display import Audio as Play
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# transformers 5 prints several deprecation warnings for Whisper's bundled generation config.
transformers.logging.set_verbosity_error()

SAMPLE_RATE = 16_000

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
# Decoding audio inside `datasets` needs the torchcodec package and FFmpeg; reading the FLAC bytes
# with soundfile avoids both.
ds = ds.cast_column("audio", Audio(decode=False))


def load_clip(i):
    """Returns (waveform as a float32 array at 16 kHz, reference transcript) for clip i."""
    example = ds[i]
    audio, sr = sf.read(io.BytesIO(example["audio"]["bytes"]), dtype="float32")
    assert sr == SAMPLE_RATE
    return audio, example["text"]


processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en").eval()
print(ds)
print(f"whisper-tiny.en: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

## Part 1: audio is just numbers

A microphone measures air pressure thousands of times per second. Each measurement is a *sample*, stored as a number between -1 and 1. Whisper expects mono audio at **16,000 samples per second (16 kHz)**, so a 5-second clip is an array of 80,000 floats.

In [ ]:
audio, reference = load_clip(0)
print(audio.shape, audio.dtype, f"min={audio.min():.2f}  max={audio.max():.2f}")
print("Reference:", reference)

seconds = np.arange(len(audio)) / SAMPLE_RATE
plt.figure(figsize=(12, 2.5))
plt.plot(seconds, audio, linewidth=0.5)
plt.xlabel("seconds")
plt.show()

Play(audio, rate=SAMPLE_RATE)

In [ ]:
# TODO(you): compute the duration in seconds of every clip in `ds` (use load_clip and SAMPLE_RATE).
# Print the shortest, the longest, and the total length in minutes.
durations = ...

In [ ]:
assert len(durations) == len(ds), "expected one duration per clip"
assert abs(sum(durations) - 481.0) < 1, "the clips should add up to about 8 minutes"
print("Correct.")

## Part 2: the log-mel spectrogram

Whisper never sees the raw waveform. Its feature extractor converts it into a **log-mel spectrogram**:

1. Slide a 25 ms window along the audio in 10 ms steps. For each window, measure how much energy there is at each frequency (a short-time Fourier transform).
2. Group those frequencies into 80 **mel** bands. The mel scale is spaced like human hearing: fine detail at low frequencies, where most speech information is, and coarse detail at high frequencies.
3. Take the **log**, because we perceive loudness roughly logarithmically, then scale the values to about -1 to 1.

Whisper always works on exactly **30 seconds** of audio: shorter clips are padded with silence and longer ones are cut off. At 10 ms per frame, that's 3,000 frames. So every input, whatever its length, becomes an 80 × 3000 array, which the model treats a bit like an image.

The longest clip you just found is 29.4 seconds, just under that limit. Your own recordings will need to stay under 30 seconds too.

In [ ]:
# TODO(you): compute Whisper's input features for `audio` with processor.feature_extractor.
# Pass sampling_rate=SAMPLE_RATE and return_tensors="pt". The call returns a dict-like object;
# keep its `input_features` tensor in `features`.
features = ...

In [ ]:
assert tuple(features.shape) == (1, 80, 3000), f"expected (1, 80, 3000), got {tuple(features.shape)}"

plt.figure(figsize=(12, 3))
plt.imshow(features[0].numpy(), origin="lower", aspect="auto", cmap="magma")
plt.xlabel("frame (10 ms each)")
plt.ylabel("mel band")
plt.colorbar()
plt.show()

**Check your understanding**

- Clip 0 is 5.9 seconds long. At roughly which frame does the speech end, and what fills the rest of the image?
- Why must the audio you fine-tune on go through exactly the same feature extractor as the audio you transcribe later?

## Part 3: tokens

Whisper's decoder writes text as **tokens**: whole words or pieces of words from a vocabulary of about 50,000 (byte-pair encoding, the same scheme GPT-2 uses). A `Ġ` at the start of a token means it begins with a space.

It also uses **special tokens** as instructions. For the English-only models (the names ending in `.en`), every transcript starts with:

- `<|startoftranscript|>`: begin a transcript.
- `<|notimestamps|>`: output text only, without timing information.

and ends with `<|endoftext|>`. Multilingual models also insert a language token such as `<|en|>` and a task token (`<|transcribe|>` or `<|translate|>`).

In [ ]:
# TODO(you): tokenize `reference` with processor.tokenizer and look at the result.
# 1. Get the token ids: calling the tokenizer on a string returns an object with `.input_ids`.
# 2. Turn the ids back into token strings with processor.tokenizer.convert_ids_to_tokens and print them.
# 3. Tokenize reference.lower() too, and compare the number of tokens with the number of words.
ids = ...

In [ ]:
assert ids[0] == processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>"), "ids should start with <|startoftranscript|>"
assert ids[-1] == processor.tokenizer.eos_token_id, "ids should end with <|endoftext|>"
print(len(reference.split()), "words,", len(ids), "tokens")

**Check your understanding**

LibriSpeech transcripts are all uppercase, and uppercase text needs about 50% more tokens than lowercase text. What would you expect to happen if you fine-tuned Whisper on these uppercase transcripts as they are? What does that tell you about how to write the transcripts for your own dataset?

## Part 4: transcribing

Whisper has two halves:

- The **encoder** reads the whole spectrogram once and turns it into 1,500 vectors, one per 20 ms of audio.
- The **decoder** writes the transcript one token at a time. At each step it looks at the encoder's vectors and at every token written so far, and outputs a score (a *logit*) for each of the ~50,000 possible next tokens. The simplest strategy, **greedy decoding**, picks the highest-scoring token, appends it, and repeats until it produces `<|endoftext|>`.

`model.generate()` runs that loop for you.

In [ ]:
# TODO(you): transcribe clip 0.
# 1. Call model.generate(features) inside `with torch.no_grad():` (inference doesn't need gradients).
# 2. Decode the output ids with processor.batch_decode(..., skip_special_tokens=True).
#    It returns one string per clip in the batch; keep the string in `hypothesis`.
hypothesis = ...

print("Reference: ", reference)
print("Hypothesis:", hypothesis)

In [ ]:
assert isinstance(hypothesis, str), "batch_decode returns a list; take its first element"
assert "quilter" in hypothesis.lower()
print("Correct.")

### Stretch: write the decoding loop yourself

This is optional, but it's the best way to understand how Whisper, and every LLM, generates text.

1. Run the encoder once: `encoder_out = model.model.encoder(features).last_hidden_state`. Check its shape.
2. Start with `tokens = torch.tensor([[sot, no_timestamps]])`.
3. In a loop, run the decoder with `model(encoder_outputs=(encoder_out,), decoder_input_ids=tokens).logits`. The result has shape (1, number of tokens so far, vocabulary size), and the last position holds the scores for the next token. Take the argmax, append it to `tokens` with `torch.cat`, and stop when it equals `processor.tokenizer.eos_token_id`.
4. Decode `tokens[0]` with `processor.tokenizer.decode(..., skip_special_tokens=True)`.

In [ ]:
# TODO(you, optional): greedy decoding by hand. Put the final text in `my_hypothesis`.
sot = processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
no_timestamps = processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")

my_hypothesis = ...

In [ ]:
assert my_hypothesis.strip() == hypothesis.strip(), f"got {my_hypothesis!r}"
print("Your loop matches generate().")

**Check your understanding**

Your loop reruns the decoder over *all* previous tokens at every step, even though only the newest one changed. How could you avoid recomputing that work? (`generate()` does this, and it's one of the main tricks behind fast LLM inference.)

## Part 5: word error rate

WER counts the smallest number of word **s**ubstitutions, **d**eletions, and **i**nsertions that turn the hypothesis into the reference, divided by the number of words **N** in the reference:

$$\text{WER} = \frac{S + D + I}{N}$$

Lower is better. 0 means perfect, and because of insertions it can go above 100%.

In [ ]:
# TODO(you): compute the WER of `hypothesis` against `reference` with jiwer.wer(reference, hypothesis).
# Then compute it again after normalizing both strings with processor.tokenizer.normalize.
raw_wer = ...
normalized_wer = ...

print(f"raw WER: {raw_wer:.0%}   normalized WER: {normalized_wer:.0%}")

The transcript is correct, yet the raw WER is 100%. Every word differs in casing or punctuation, and "Mr." isn't "MISTER". That's why you always **normalize both sides before scoring**. `normalize` is the English normalizer from the Whisper paper: it lowercases, removes punctuation, and standardizes spellings, numbers, and abbreviations.

To score a whole dataset, use **corpus-level WER**: add up the errors over all clips and divide by the total number of reference words. That's what `jiwer.wer` computes when you pass it two lists. It isn't the same as averaging per-clip WERs, which would let a two-word clip with one mistake (50%) count as much as a 40-word clip.

In [ ]:
# TODO(you): evaluate whisper-tiny.en on all 73 clips.
# 1. Write transcribe_batch(model, audios), which turns a list of waveforms into a list of transcripts.
#    The feature extractor, generate(), and batch_decode() all accept batches.
# 2. Loop over the dataset in batches of 8, collecting references and hypotheses.
# 3. Compute the corpus-level WER on the normalized texts.
def transcribe_batch(model, audios):
    ...


references, hypotheses = [], []
...

corpus_wer = ...
print(f"WER: {corpus_wer:.1%}")

In [ ]:
assert len(hypotheses) == len(ds), "expected one hypothesis per clip"
assert 0.07 < corpus_wer < 0.11, "whisper-tiny.en should score about 9% here; check your normalization"
print("Correct.")

In [ ]:
# TODO(you): error analysis. Compute the WER of each clip on its own, sort the clips from worst to best,
# and print the five worst with their reference and hypothesis side by side.
# What kinds of words does the model get wrong?


Most of the worst errors are names and rare words the model barely saw in training, like "Kaliko" and "whirr". Those are exactly the errors fine-tuning on your own voice and vocabulary can fix, and error analysis like this is how you'll find them in your own data.

### Compare model sizes

Bigger models are usually more accurate and always slower. Wrap your evaluation code in a function `evaluate(model)` that returns `(wer, seconds)`, then run it on `openai/whisper-base.en`. The processor is the same for both models, so only the model changes.

| model | parameters | WER | seconds for 73 clips |
|---|---|---|---|
| whisper-tiny.en | | | |
| whisper-base.en | | | |

In [ ]:
# TODO(you): evaluate both models and fill in the table above.
# base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base.en").eval()


## What's next

In notebook 2 you'll record your own dataset: about 200 sentences in your voice, mixing everyday phrases with vocabulary you actually use (names, tools, jargon). You'll measure Whisper's WER on your voice, and that number becomes the baseline fine-tuning has to beat.

Before moving on, make sure you can answer:

1. Why does every input become 80 × 3000, and what fills the unused part?
2. Which special tokens start an English-only transcript, and what does each one do?
3. Why is raw WER meaningless without normalization?
4. What does the decoder output at each step, and how does greedy decoding use it?